# PREMA: Prediction of Metabolic Acidosis in Preterm Infants

This notebook demonstrates how to apply the PREMA model to predict metabolic acidosis (pH < 7.2 and BE < −7) using 1-minute interval vital sign data (HR, SpO₂) from preterm infants.

### Input data (`vs_sample.csv`)
| Column | Description |
|---|---|
| `patient_id` | Patient identifier |
| `birthdate` | Date of birth |
| `GA` | Gestational age at birth (days) |
| `BW` | Birth weight (kg) |
| `male` | Sex (1 = male, 0 = female) |
| `Timestamp` | Measurement date and time (1-min interval) |
| `HR` | Heart rate (bpm) |
| `sat` | Oxygen saturation / SpO₂ (%) |

### Output
For each 4-hour window (sliding by 30-min steps), the model outputs the predicted probability of metabolic acidosis.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import joblib
from datetime import timedelta
from scipy import stats
import antropy as ant
import warnings
from tqdm import tqdm

warnings.filterwarnings('ignore', message='Precision loss occurred in moment calculation')

## 2. Load model

In [ ]:
model = joblib.load('PREMA.pkl')
print('Model loaded successfully.')

## 3. Load data

In [ ]:
data = pd.read_csv('vs_sample.csv')

print(f'Loaded {len(data):,} records | {data["patient_id"].nunique()} patient(s)')
data.head()

## 4. Feature extraction functions

In [ ]:
def extract_static_features(patient_data, group_start):
    first = patient_data.iloc[0]
    birth_date = pd.to_datetime(first['birthdate'])
    day_value = (group_start - birth_date).days
    return {
        'GA'  : first['GA'],
        'BW'  : first['BW'],
        'male': first['male'],
        'PMA' : first['GA'] + day_value   # postmenstrual age (days)
    }


def extract_time_features(obs_window_start, group_start, prefix=''):
    event_time = group_start + pd.Timedelta(hours=10)
    delta_t = (obs_window_start - event_time).total_seconds() / 60
    hour = obs_window_start.hour
    dow  = obs_window_start.dayofweek
    return {
        f'{prefix}delta_t_minutes': delta_t,
        f'{prefix}hour'           : hour,
        f'{prefix}hour_sin'       : np.sin(2 * np.pi * hour / 24),
        f'{prefix}hour_cos'       : np.cos(2 * np.pi * hour / 24),
        f'{prefix}dayofweek'      : dow,
        f'{prefix}dayofweek_sin'  : np.sin(2 * np.pi * dow / 7),
        f'{prefix}dayofweek_cos'  : np.cos(2 * np.pi * dow / 7),
    }


def missing_ratio(data, n_points=120):
    return (data.isnull().sum() / n_points) * 100


def extract_vital_features(vital_data, vs):
    arr = vital_data.values.astype(float)
    clean = arr[~np.isnan(arr)]
    feats = {
        f'{vs}_mean'  : np.nanmean(arr)   if len(arr) > 0 else np.nan,
        f'{vs}_std'   : np.nanstd(arr)    if len(arr) > 0 else np.nan,
        f'{vs}_mad'   : np.mean(np.abs(arr - np.nanmean(arr))),
        f'{vs}_median': np.nanmedian(arr),
        f'{vs}_idr'   : np.nanpercentile(arr, 90) - np.nanpercentile(arr, 10) if len(arr) > 0 else np.nan,
        f'{vs}_kurt'  : stats.kurtosis(clean) if len(clean) > 0 else np.nan,
        f'{vs}_skew'  : stats.skew(clean)     if len(clean) > 0 else np.nan,
    }
    try:
        feats[f'{vs}_SampEn'] = ant.sample_entropy(clean) if len(clean) > 2 else np.nan
    except Exception:
        feats[f'{vs}_SampEn'] = np.nan
    return feats


def extract_feature(patient_data, group_data, group_start):
    vital_signs = ['HR', 'sat']
    static_features = extract_static_features(patient_data, group_start)
    all_timestep_features = {}

    for i in range(5):  # 5 overlapping 2-h windows, each shifted by 30 min
        prefix = f't{i}_'
        obs_start = group_start + pd.Timedelta(minutes=30 * i)
        obs_end   = obs_start + pd.Timedelta(hours=2) - pd.Timedelta(minutes=1)
        obs_data  = group_data[
            (group_data['date_time'] >= obs_start) &
            (group_data['date_time'] <= obs_end)
        ]

        timestep_feats = extract_time_features(obs_start, group_start, prefix=prefix)

        if obs_data.empty or obs_data[vital_signs].isnull().all().all():
            for vs in vital_signs:
                timestep_feats[f'{prefix}missing_{vs}'] = 100.0
                for ft in ['mean', 'std', 'mad', 'median', 'idr', 'kurt', 'skew', 'SampEn']:
                    timestep_feats[f'{prefix}{vs}_{ft}'] = np.nan
        else:
            for vs in vital_signs:
                vital_data = obs_data[vs].dropna()
                timestep_feats[f'{prefix}missing_{vs}'] = missing_ratio(vital_data, n_points=120)
                if len(vital_data) > 0:
                    extracted = extract_vital_features(vital_data, vs)
                    for ft, val in extracted.items():
                        timestep_feats[f'{prefix}{ft}'] = val
                else:
                    for ft in ['mean', 'std', 'mad', 'median', 'idr', 'kurt', 'skew', 'SampEn']:
                        timestep_feats[f'{prefix}{vs}_{ft}'] = np.nan

        all_timestep_features.update(timestep_feats)

    return pd.DataFrame([{**static_features, **all_timestep_features}])

## 5. Run prediction

In [ ]:
def run_prema(data, model):
    """
    Slide a 4-hour window over each patient's vital sign time series (30-min step)
    and return the PREMA model's predicted probability for each window.
    """
    data = data.copy()
    data['date_time'] = pd.to_datetime(data['date_time'])
    data['birthdate'] = pd.to_datetime(data['birthdate'])
    data = data.sort_values(['patient_id', 'date_time']).reset_index(drop=True)

    results = []

    for pid in tqdm(data['patient_id'].unique(), desc='Patients'):
        patient_data = data[data['patient_id'] == pid].copy()
        start_time   = patient_data['date_time'].min()
        end_time     = patient_data['date_time'].max()

        group_start = start_time
        group_id    = 0

        while group_start + pd.Timedelta(hours=4) - pd.Timedelta(minutes=1) <= end_time:
            group_end  = group_start + pd.Timedelta(hours=4) - pd.Timedelta(minutes=1)
            group_data = patient_data[
                (patient_data['date_time'] >= group_start) &
                (patient_data['date_time'] <= group_end)
            ]

            features_df = extract_feature(patient_data, group_data, group_start)
            features_df = features_df.apply(pd.to_numeric, errors='coerce')
            features_df = features_df.replace([np.inf, -np.inf], np.nan)

            prob = model.predict_proba(features_df)[0, 1]

            results.append({
                'patient_id'      : pid,
                'group_id'        : group_id,
                'window_start'    : group_start,
                'window_end'      : group_end,
                'predicted_prob'  : prob,
            })

            group_start += pd.Timedelta(minutes=30)
            group_id    += 1

    return pd.DataFrame(results)


result = run_prema(data, model)
print(f'Total windows predicted: {len(result):,}')
result.head(10)

## 6. Save results

In [ ]:
result.to_csv('prema_predictions.csv', index=False)
print('Results saved to prema_predictions.csv')